# 02 — Calidad y Limpieza

**Objetivo:** Corregir todos los problemas detectados en el notebook 01, **documentando cada paso** en una tabla de auditoría que se guarda en `logs/pipeline_log.csv` (qué se hizo, cuántas filas quedaron, cuántos nulos hay). Al final se exporta el dataset limpio a `data/processed/streaming_users_clean.csv`.

### Criterio de tratamiento (justificado)
- **Duplicados** → eliminar (clave `user_id` debe ser única).
- **Valores imposibles por rango** (edades fuera de [13,100], tiempos negativos, tickets -1) → convertir a nulo e imputar.
- **Valores imposibles de relleno** (99999/50000 en watch_time; 99/150 en tickets): aparecen repetidos y separados por un hueco del resto de la distribución → son códigos de relleno por error de carga, **no** outliers reales → a nulo e imputar.
- **Outliers reales** (cola alta legítima de watch_time) → **winsorizar** (acotar), no eliminar.
- **Categóricas** → normalizar a categorías canónicas.
- **Fechas** → parsear formatos válidos; lo no recuperable o futuro → nulo (sin imputar, para no inventar).

## Preparación: carga y sistema de auditoría

In [1]:
import pandas as pd
import numpy as np
import os

# Cargamos el crudo y preservamos el original inmutable
df_raw = pd.read_json('../data/raw/streaming_users_dirty.json')
df = df_raw.copy()

N0 = len(df)
log = []
def auditar(paso, desc):
    log.append({'Paso': paso, 'Descripción': desc,
                'Filas': len(df), 'Nulos': int(df.isnull().sum().sum()),
                'Retención (%)': round(len(df)/N0*100, 2)})

auditar(0, 'Estado original')
print(f"Original: {len(df)} filas, {int(df.isnull().sum().sum())} nulos")

Original: 8160 filas, 753 nulos


## Paso 1 — Eliminar duplicados

In [2]:
df = df.drop_duplicates()                                  # filas 100% idénticas
df = df.drop_duplicates(subset='user_id', keep='first')    # user_id único
auditar(1, 'Eliminar duplicados')
print("Filas:", len(df), "| user_id único:", df['user_id'].is_unique)

Filas: 8000 | user_id único: True


## Paso 2 — Normalizar variables categóricas

In [3]:
def limpiar(s): return s.astype(str).str.strip().str.lower()

plan_map = {'básico':'Básico','basico':'Básico','basic':'Básico',
            'estándar':'Estándar','estandar':'Estándar','std':'Estándar','standard':'Estándar',
            'premium':'Premium','premiun':'Premium'}
df['subscription_plan'] = limpiar(df['subscription_plan']).map(plan_map)

country_map = {'argentina':'Argentina','arg':'Argentina','brasil':'Brasil','brazil':'Brasil','bra':'Brasil',
               'chile':'Chile','chl':'Chile','colombia':'Colombia','col':'Colombia',
               'méxico':'México','mexico':'México','mex':'México','perú':'Perú','peru':'Perú','per':'Perú',
               'uruguay':'Uruguay','ury':'Uruguay'}
df['country'] = limpiar(df['country']).map(country_map)

genre_map = {'acción':'Acción','accion':'Acción','action':'Acción','comedia':'Comedia','comedy':'Comedia',
             'crime':'Crime','crimen':'Crime','documental':'Documental','documentary':'Documental','doc':'Documental',
             'drama':'Drama','romance':'Romance','thriller':'Thriller','thriler':'Thriller'}
serie_g = limpiar(df['favorite_genre'])
df['favorite_genre'] = serie_g.where(serie_g != 'nan').map(genre_map)

auditar(2, 'Normalizar categóricas')
print("Planes:", sorted(df['subscription_plan'].dropna().unique()))
print("Países:", sorted(df['country'].dropna().unique()))
print("Géneros:", sorted(df['favorite_genre'].dropna().unique()))

Planes: ['Básico', 'Estándar', 'Premium']
Países: ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']
Géneros: ['Acción', 'Comedia', 'Crime', 'Documental', 'Drama', 'Romance', 'Thriller']


**Qué se hizo:** se pasó todo a minúsculas, se quitaron espacios y se mapeó cada variante a una única categoría canónica. Así `básico`, `BASICO` y `Basic` quedan todos como `Básico`. Los textos `nan` literales se convierten en nulo real para imputarlos después.

## Paso 3 — Convertir valores imposibles a nulo

In [4]:
# age fuera de [13, 100]
m_age = (df['age'] < 13) | (df['age'] > 100)
df.loc[m_age, 'age'] = np.nan

# watch_time negativos y valores imposibles (>=10000)
m_wt = (df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] >= 10000)
df.loc[m_wt, 'monthly_watch_time_mins'] = np.nan

# tickets negativos y valores imposibles (99, 150)
m_tk = (df['customer_support_tickets'] < 0) | (df['customer_support_tickets'].isin([99, 150]))
df.loc[m_tk, 'customer_support_tickets'] = np.nan

print(f"age imposibles -> nulo: {m_age.sum()}")
print(f"watch_time inválidos -> nulo: {m_wt.sum()}")
print(f"tickets inválidos -> nulo: {m_tk.sum()}")
auditar(3, 'Valores imposibles -> nulo')

age imposibles -> nulo: 120
watch_time inválidos -> nulo: 80
tickets inválidos -> nulo: 96


**Por qué imposible de relleno y no outlier:** los valores 99999 (watch_time) y 99/150 (tickets) aparecen **repetidos exactamente** y **separados por un hueco** del resto de la distribución. Si fueran medidas reales habría valores intermedios. Ese patrón delata códigos de relleno por error de carga, así que se tratan como dato faltante, no como cola de la distribución.

## Paso 4 — Parsear y validar fechas

In [5]:
fechas = pd.to_datetime(df['last_login_date'], errors='coerce', format='mixed', dayfirst=False)
fechas = fechas.where(fechas <= pd.Timestamp('2026-06-24'))   # descartar futuras
df['last_login_date'] = fechas
print("Fechas válidas:", df['last_login_date'].notna().sum())
print("Fechas nulas (no recuperables):", df['last_login_date'].isna().sum())
auditar(4, 'Parsear/validar fechas')

Fechas válidas: 7601
Fechas nulas (no recuperables): 399


## Paso 5 — Winsorizar outliers reales de watch_time

In [6]:
Q1, Q3 = df['monthly_watch_time_mins'].quantile([0.25, 0.75])
limite = Q3 + 3.0 * (Q3 - Q1)        # criterio IQR k=3
n_out = (df['monthly_watch_time_mins'] > limite).sum()
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].clip(upper=limite)
print(f"Límite k=3: {limite:.1f} | valores acotados: {n_out}")
auditar(5, f'Winsorizar watch_time (k=3, lim={limite:.0f})')

Límite k=3: 2692.6 | valores acotados: 108


**Por qué winsorizar y no eliminar:** una vez quitados los valores imposibles, la cola alta de watch_time corresponde a **usuarios reales de alto consumo**. Eliminarlos perdería información válida; acotarlos al límite k=3 conserva la fila y limita la distorsión.

### Diagnóstico del mecanismo de faltantes (¿MCAR o MAR?)

Antes de imputar, verificamos **cómo** se distribuyen los faltantes. Si aparecieran al azar por igual en todos los grupos, el mecanismo sería **MCAR** (Missing Completely At Random) y bastaría una mediana global. Si en cambio la tasa de faltantes **difiere según el plan**, el mecanismo es **MAR** (Missing At Random, asociado a una variable observable) y corresponde imputar **por grupo**. Esta es la evidencia que justifica la decisión, en lugar de elegir el método de antemano.

In [7]:
# Marcamos qué filas tienen faltante en cada variable a imputar
df['_falta_watch'] = df['monthly_watch_time_mins'].isna()
df['_falta_genre'] = df['favorite_genre'].isna()

# Tasa de faltantes (%) por plan de suscripción
tasa = pd.DataFrame({
    'watch_time (% nulos)': df.groupby('subscription_plan')['_falta_watch'].mean().mul(100).round(2),
    'favorite_genre (% nulos)': df.groupby('subscription_plan')['_falta_genre'].mean().mul(100).round(2),
})
print(tasa)

# Limpiamos las columnas auxiliares
df = df.drop(columns=['_falta_watch', '_falta_genre'])

                   watch_time (% nulos)  favorite_genre (% nulos)
subscription_plan                                                
Básico                             1.08                      3.14
Estándar                           2.27                      3.02
Premium                           10.74                      2.65


**Interpretación:** la tasa de faltantes en `watch_time` **crece fuertemente con el plan** — Básico ≈ 1,1%, Estándar ≈ 2,3% y **Premium ≈ 10,7%** (casi 10 veces más que Básico). Esa diferencia marcada confirma un mecanismo **MAR**: la ausencia del dato está asociada al plan, una variable observable. Por eso imputamos con la **mediana por plan** y no global: como Premium concentra a los usuarios de mayor consumo, usar la mediana global (dominada por el plan Básico, mayoritario) rellenaría los huecos de Premium con valores demasiado bajos, sesgando el análisis. La imputación por grupo evita ese sesgo. En cambio, `favorite_genre` tiene faltantes parejos (~3% en todos los planes, más cercano a MCAR); al ser categórica se imputa por moda.

## Paso 6 — Imputar valores faltantes

In [8]:
# Numéricas -> mediana POR PLAN (el comportamiento depende del plan -> mecanismo MAR)
for col in ['age', 'monthly_watch_time_mins']:
    df[col] = df.groupby('subscription_plan')[col].transform(lambda x: x.fillna(x.median()))

df['customer_support_tickets'] = (df.groupby('subscription_plan')['customer_support_tickets']
                                    .transform(lambda x: x.fillna(x.median())).round().astype(int))

# Categórica -> moda global
df['favorite_genre'] = df['favorite_genre'].fillna(df['favorite_genre'].mode()[0])

# last_login_date: NO se imputa (inventar una fecha de login sería un dato falso)
auditar(6, 'Imputar faltantes (excepto fecha)')
print("Nulos restantes por columna:")
print(df.isnull().sum())

Nulos restantes por columna:
user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
last_login_date             399
customer_support_tickets      0
dtype: int64


**Justificación de la imputación:**
- `age`, `watch_time`, `tickets` → **mediana por plan** (no global): el consumo y el perfil dependen del plan contratado, así que el grupo condiciona el valor. La mediana es robusta ante asimetría.
- `favorite_genre` → **moda** (es categórica).
- `last_login_date` → **se deja como nulo a propósito**: rellenar una fecha de último acceso con un valor inventado sesgaría cualquier análisis de actividad.

## Guardar log de auditoría y dataset limpio

In [9]:
# Log de pipeline -> logs/pipeline_log.csv (columnas exactas que pide la consigna)
os.makedirs('../logs', exist_ok=True)
log_df = pd.DataFrame(log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)
print("Log guardado en logs/pipeline_log.csv:\n")
print(log_df.to_string(index=False))

Log guardado en logs/pipeline_log.csv:

 Paso                           Descripción  Filas  Nulos  Retención (%)
    0                       Estado original   8160    753         100.00
    1                   Eliminar duplicados   8000    753          98.04
    2                Normalizar categóricas   8000    753          98.04
    3            Valores imposibles -> nulo   8000   1049          98.04
    4                Parsear/validar fechas   8000   1128          98.04
    5 Winsorizar watch_time (k=3, lim=2693)   8000   1128          98.04
    6     Imputar faltantes (excepto fecha)   8000    399          98.04


In [10]:
# Dataset limpio -> data/processed/streaming_users_clean.csv (entregable de la consigna)
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print("Guardado:")
print(" - data/processed/streaming_users_clean.csv")
print("Filas finales:", len(df), "| Retención:", round(len(df)/N0*100,2), "%")

Guardado:
 - data/processed/streaming_users_clean.csv
Filas finales: 8000 | Retención: 98.04 %


## Verificación final

In [11]:
print("user_id único:", df['user_id'].is_unique)
print("age rango:", df['age'].min(), "-", df['age'].max())
print("watch_time negativos:", (df['monthly_watch_time_mins']<0).sum(), "| máx:", round(df['monthly_watch_time_mins'].max(),1))
print("tickets negativos:", (df['customer_support_tickets']<0).sum(), "| máx:", df['customer_support_tickets'].max())

print("\nCategóricas normalizadas:")
for col in ['subscription_plan', 'country', 'favorite_genre']:
    print(f"  {col}: {df[col].nunique()} categorías -> {sorted(df[col].dropna().unique())}")

print("\nNulos restantes (solo fecha esperado):")
print(df.isnull().sum())

user_id único: True
age rango: 13.0 - 80.0
watch_time negativos: 0 | máx: 2692.6
tickets negativos: 0 | máx: 5

Categóricas normalizadas:
  subscription_plan: 3 categorías -> ['Básico', 'Estándar', 'Premium']
  country: 7 categorías -> ['Argentina', 'Brasil', 'Chile', 'Colombia', 'México', 'Perú', 'Uruguay']
  favorite_genre: 7 categorías -> ['Acción', 'Comedia', 'Crime', 'Documental', 'Drama', 'Romance', 'Thriller']

Nulos restantes (solo fecha esperado):
user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0
last_login_date             399
customer_support_tickets      0
dtype: int64
